In [20]:
# Load the file and create a list of street names
file_path = 'streets_names.txt'  # Change this if the path is different
with open(file_path, 'r') as file:
    # Read and split by newlines, then clean each name
    streets = [name.strip().replace("'", "") for name in file.read().splitlines()]
print(streets)


['Asfur', 'E Hillel', 'ESaan', 'EdEd', 'Edsh Shafik', 'Eko', 'Ekrmh Ben Avi GHl', 'Ekrmh Ben GHl Sam1', 'El Prsht Drkym', 'Ela Asyuti', 'Elony Nch', 'Elr', 'Ely Avi Talv', 'Ely Ibn Abu Talv', 'Ely Mohr', 'Em OEolmo', 'Emnoal Zamir', 'Emnoal Zysamn', 'Emon', 'Emr', 'Emy', 'Emyr', 'Emyrn Emnoal', 'Enath Hahadasha', 'Entyn', 'Envr', 'Eogv', 'Eoly Hagardom', 'Eomr', 'Eomr Al ChYam', 'Eomr Ben Al ChTav', 'Eomr Ben Al Chtav', 'Eomr Ben Al-Chtav', 'Eomr Dvora', 'Eomr Ibn Al-Easa', 'Eomr Ibn Al-Chtav', 'Eomr Ibn Ktav', 'Eomrym', 'Eonot Shnh', 'Eooysa', 'Eooysa Sam2', 'Eooysat', 'Eopry SeAdya', 'Eopryt', 'Eoqvh Ben NapE', 'Eorqvy Tsdoq', 'Eosapyh', 'EotMn Ben Epan', 'EotMn Ben Epan Sam6', 'Eprh', 'Eprh Chzh', 'Eq Al Asaalh', 'Eq Al Chlydyh', 'Eq Shych Rychan', 'Eqvt A Chnqh', 'Eqvt A Mopty', 'Eqvt A Sarayh', 'Eqvt A Tut', 'Eqvt A Vtych', 'Eqvt Al-GVl', 'Eqvt Chv Romain', 'Eqvt Drooysh', 'Eqvt Rysaasa', 'Eqvt Shadd', 'Eqvt Shych Hassan', 'Eqvt Shych Lulu', 'Eqvt Tqyh', 'ErEr', 'Eraq Al Tayrh', '

In [45]:
import re
import string

def preprocess_sentence(sentence):
    global streets

    lower_streets = [street.lower() for street in streets]

    sentence = re.sub(r'(\d+)([a-zA-Z]+)', r'\1 \2', sentence)
    # Convert to lowercase
    sentence = sentence.lower()

    sentence = re.sub(r'\bmy\b', "your", sentence)
    # Remove punctuation
    sentence = sentence.translate(str.maketrans('', '', string.punctuation))

    # Tokenize the sentence
    tokens = sentence.split()
    i = 0
    while i < len(tokens):
        for length in range(1, len(tokens) - i + 1):  # Check word combinations of length 1 to len(sentence)
            word_sequence = ' '.join(tokens[i:i+length])
            if word_sequence in lower_streets:  # If the sequence is found in streets
                tokens[i:i+length] = [word_sequence]  # Merge the sequence into one token
                i += length - 1  # Skip over the words that have been merged
                break
        i += 1  # Continue to the next token

    # Remove empty tokens
    tokens = [token for token in tokens if token]
    return tokens

# Example
sentence = "Show me a running path from downtown to ekrmh ben avi ghl?"
tokens = preprocess_sentence(sentence)
print(tokens)


['show', 'me', 'a', 'running', 'path', 'from', 'downtown', 'to', 'ekrmh ben avi ghl']


In [22]:
start_location_keywords = {
    "from"
    #,"at"
}

end_location_keywords = {
    "to",
    "at"
}

difficulty_keywords = {
    "easy",
    "moderate",
    "hard",
    "beginner",
    "intermediate",
    "advanced",
    "simple",
    "basic",
    "challenging",
    "difficult",
    "entrylevel",
    "novice",
    "expert",
    "strenuous",
    "elementary",
    "complex",
    "very",
    "difficult"
}

In [46]:
def ends_with_sentence(tokens, bio_tags):
    sentence_ends = [
        {"phrase": "bro", "word_count": 1},
        {"phrase": "bruh", "word_count": 1},
        {"phrase": "my man", "word_count": 2},
        {"phrase": "my sister", "word_count": 2},
        {"phrase": "my brother", "word_count": 2},
        {"phrase": "plz", "word_count": 1},
        {"phrase": "thx", "word_count": 1},
        {"phrase": "Please", "word_count": 1},
        {"phrase": "Thanks", "word_count": 1},
        {"phrase": "Thank you", "word_count": 2},
        {"phrase": "If you could", "word_count": 3},
        {"phrase": "If you don't mind", "word_count": 4},
        {"phrase": "When you get a chance", "word_count": 5},
        {"phrase": "At your earliest convenience", "word_count": 4},
        {"phrase": "I'd appreciate it", "word_count": 3},
        {"phrase": "If possible", "word_count": 2},
        {"phrase": "Kindly", "word_count": 1},
        {"phrase": "If you would", "word_count": 3},
        {"phrase": "Much appreciated", "word_count": 2},
        {"phrase": "Looking forward to it", "word_count": 4},
        {"phrase": "Whenever you can", "word_count": 3},
        {"phrase": "I'd be grateful", "word_count": 3}
    ]

    for entry in sentence_ends:
        phrase = entry["phrase"].lower().split()  # Convert phrase to lowercase and split into tokens
        word_count = entry["word_count"]

        # Check if the last `word_count` tokens of the sentence match the phrase
        if len(tokens) >= word_count and tokens[-word_count:] == phrase:
            # Mark those tokens as 'O' in the bio_tags
            for i in range(word_count):
                bio_tags['O'].append(tokens[-word_count + i])

            # Shorten the tokens list by removing the marked tokens
            tokens = tokens[:-word_count]
            return tokens, bio_tags

    # If no phrase is matched, return the original tokens and bio_tags
    return tokens, bio_tags

def a_func(lst_a, lst_b):
  for word in lst_a:
    if word in lst_b:
      return True
  return False

def tag_bio(start_location_keywords, end_location_keywords, difficulty_keywords, streets, tokens):
    # Initialize the dictionary to store BIO tags
    bio_tags = {
        'B-loca_start_num': [],
        'I-start_location': [],
        'B-end_location': [],
        'I-difficulty': [],
        'B-route_length': [],
        'B-start_location': [],
        'I-route_length': [],
        'B-loca_end_num': [],
        'I-end_location': [],
        'B-difficulty': [],
        'O': []
    }

     # Flags for state tracking
    in_start_location = False
    in_end_location = False
    in_difficulty = False
    digit_found = False

    i = 0
    tokens, bio_tags = ends_with_sentence(tokens, bio_tags)

    while i < len(tokens):
        token = tokens[i]

        # Rule for B-loca_start_num: numeric token at the start of a location
        if token.isdigit():
          if i+1 < len(tokens) and tokens[i+1] in ["km", "kilometers", "meters", "miles"]:
            bio_tags['B-route_length'].append(token)
          elif not digit_found and not a_func(tokens[:i], end_location_keywords):
            bio_tags['B-loca_start_num'].append(token)
            digit_found = True
          else:
            bio_tags['B-loca_end_num'].append(token)

        elif "and" == token:
          bio_tags['O'].append(token)

        elif i+1 < len(tokens) and token == "to" and tokens[i+1] not in [street.lower() for street in streets]:
          bio_tags['O'].append(token)
          bio_tags['O'].append(tokens[i+1])
          i += 1

        elif token in difficulty_keywords and not in_difficulty and not bio_tags['B-difficulty']:
            bio_tags['B-difficulty'].append(token)
            in_difficulty = True

        elif token in difficulty_keywords and in_difficulty and bio_tags['B-difficulty'] and tokens[i-1] in bio_tags['B-difficulty']+bio_tags['I-difficulty']:
            bio_tags['I-difficulty'].append(token)

        # Rule for B-start_location: first token in a location
        elif i-1 >= 0 and re.match(r"^[a-z]+(?: [a-z]+)*$", token) and tokens[i-1] in start_location_keywords and not in_start_location and not bio_tags['B-start_location'] or (i-2>=0 and tokens[i-2] == "starting" and tokens[i-1] == "at"):
          bio_tags['B-start_location'].append(token)
          in_start_location = True

        # Rule for I-start_location: subsequent tokens in the location name
        elif i - 1 >= 0 and in_start_location and re.match(r"^[a-z]+$", token) and not re.match(r"^[a-z] +$", tokens[i-1]) and tokens[i-1] in bio_tags['B-start_location']+bio_tags['I-start_location'] and token not in set(list(start_location_keywords)+list(end_location_keywords)):
          bio_tags['I-start_location'].append(token)

        # Rule for B-end_location: first token after 'to' or 'at'
        elif i - 1 >= 0 and re.match(r"^[a-z]+(?: [a-z]+)*$", token) and tokens[i-1] in end_location_keywords and not in_end_location and not bio_tags['B-end_location'] or (i-2>=0 and tokens[i-2] == "ending" and tokens[i-1] == "at"):
          bio_tags['B-end_location'].append(token)
          in_end_location = True

        # Rule for I-end_location: subsequent tokens in the location name
        elif i - 1 >= 0 and in_end_location and re.match(r"^[a-z]+$", token) and not re.match(r"^[a-z] +$", tokens[i-1]) and tokens[i-1] in bio_tags['B-end_location']+bio_tags['I-end_location'] and token not in set(list(start_location_keywords)+list(end_location_keywords)):
          bio_tags['I-end_location'].append(token)

        # Rule for O: any token not matching other tags
        else:
          bio_tags['O'].append(token)

        i += 1

    return bio_tags


# Example
sentence = "I would like to run 5 km starting from my current location to hanevim 7 and make it very difficult."

sentences = [
    "i want to generate a route. 10 km long, starting at my location and very hard. should end at haneviim 37. lets go!",
    "from my current location, thx! :)",
    "Show me a running path from downtown to ekrmh ben avi ghl",
    "I want to go for a 3km easy run starting from Edsh Shafik to Eko eko",
    "Plan a moderate difficulty route from my house to Elr that's about 7km",
    "Need a challenging 10km loop beginning from Elr",
    "Create a beginner-friendly 2km route starting from the train station",
    "I'd like to run from the coffee shop to the library, make it hard",
    "Plan a 5k route departing from the community center"
    "Looking for a moderate trail run starting at the forest entrance to the lake",
    "Map out an advanced 8km path from my office to the park",
    "Need a running route from the school to the sports complex, around 4km",
    "Create a strenuous route beginning at the plaza ending at the hill viewpoint",
    "Show me an easy running path that starts and ends at the mall",
]


def print_bio_tags(bio_tags):
  for key, value in bio_tags.items():
        if value:  # Check if the list is not empty
            print(f"{key}: {value}")

# tokens = preprocess_sentence(sentence)
# bio_tags = tag_bio(start_location_keywords, end_location_keywords, difficulty_keywords, streets, tokens)

In [47]:
for sentence in sentences[:1]:
    print(sentence)
    tokens = preprocess_sentence(sentence)
    bio_tags = tag_bio(start_location_keywords, end_location_keywords, difficulty_keywords, streets, tokens)
    print_bio_tags(bio_tags)
    print("---"*100)

i want to generate a route. 10 km long, starting at my location and very hard. should end at haneviim 37. lets go!
I-start_location: ['location']
B-end_location: ['haneviim']
I-difficulty: ['hard']
B-route_length: ['10']
B-start_location: ['your']
B-loca_end_num: ['37']
B-difficulty: ['very']
O: ['i', 'want', 'to', 'generate', 'a', 'route', 'km', 'long', 'starting', 'at', 'and', 'should', 'end', 'at', 'lets', 'go']
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
